# You need
- OPENAI_API_KEY
- COHERE_API_KEY
- MISTRAL_API_KEY



# Strategy

- select LLM (`ChatOpenAI` or `ChatMistralAI`)
- select embeddings for reranking strategy (`OpenAIEmbeddings, OllamaEmbeddings, MistralAIEmbeddings, DatabricksEmbeddings`)
- select chuncking method (`pdf page, RecursiveCharacterTextSplitter, CharacterTextSplitter, SemanticChunker`)
- select reranking strategy (`None, CohereRerank, FlashrankRerank, CrossEncoderReranker, LLMListwiseRerank`)
- Evaluation methods (`GEval, FaithfulnessMetric, ContextualRelevancyMetric, AnswerRelevancyMetric`)


I have created a class, where you just define any combinations of the above setting and you can quickly run the questions.

I have a few sections at the end with their evaluation: Config1, Config2, Config3, Config4, ...


You can use the following functions to select differnt settings:
```
        1- set_llm
        2- set_embeddings
        3- set_text_splitter
        4- rerank_method
```

In [1]:
import logging
import httpx

logging.basicConfig(
    level=logging.WARNING
)


In [2]:
# to fix Chroma error with sqlite3
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')


In [3]:
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI


from langchain_chroma import Chroma
# we can also use FAISS
#from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_mistralai import MistralAIEmbeddings
#from langchain_databricks import DatabricksEmbeddings # I got error in versions

from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_experimental.text_splitter import SemanticChunker


In [4]:
from langchain_community.document_loaders import PyPDFLoader

In [5]:
# reranke
from langchain.retrievers import ContextualCompressionRetriever
#from langchain.retrievers.contextual_compression import ContextualCompressionRetriever


from langchain.retrievers.document_compressors import CohereRerank

from langchain.retrievers.document_compressors import FlashrankRerank

from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank


In [6]:
# evaluate
from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

In [7]:
class QA:
    def __init__(self, fname_pdf):
        self.text_splitter = None
        self.rerank_method = None
        self.rag_chain = None
        self.retriever = None
        self.gpt_model = "gpt-4o-mini"
        #####
        loader = PyPDFLoader(fname_pdf)
        self.docs = loader.load() # converts to langchain_core.documents.base.Document
        print("number of pages:", len(self.docs))
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
        self.llm = ChatOpenAI(model=self.gpt_model)
        system_prompt = (
            "You are an assistant for question-answering tasks. "
            "Use the following pieces of retrieved context to answer "
            "the question. If you don't know the answer, say that you "
            "don't know."
            "\n\n"
            "{context}"
        )
        self.prompt = ChatPromptTemplate.from_messages(
            [("system", system_prompt),
             ("human", "{input}")]
        )


    def set_llm(self, llm):
        # default is self.llm = ChatOpenAI(model=self.gpt_model)
        self.llm = llm

    def set_embeddings(self, embeddings):
        """
        default:
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
        """
        self.embeddings = embeddings

    def set_text_splitter(self, text_splitter):
        self.text_splitter = text_splitter
    

    def fill_vectorstore(self):
        vectorstore = Chroma.from_documents(documents=self.splits, embedding=self.embeddings)
        # FAISS is another option
        self.retriever = vectorstore.as_retriever()


    def compute_rag_chain(self):
        print("rerank_method:", self.rerank_method)
        question_answer_chain = create_stuff_documents_chain(self.llm, self.prompt)
        if self.rerank_method == "CohereRerank":
            compressor = CohereRerank(model="rerank-english-v3.0")
            compression_retriever = ContextualCompressionRetriever(
                base_compressor=compressor, base_retriever=self.retriever
            )
            self.rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)
        elif self.rerank_method == "FlashrankRerank":
            compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2")
            compression_retriever = ContextualCompressionRetriever(
                base_compressor=compressor, base_retriever=self.retriever
            )
            self.rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)
        elif self.rerank_method == "CrossEncoderReranker":
            model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
            compressor = CrossEncoderReranker(model=model, top_n=3)
            compression_retriever = ContextualCompressionRetriever(
                base_compressor=compressor, base_retriever=self.retriever
            )
            self.rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)
        elif self.rerank_method == "LLMListwiseRerank":
            compressor = LLMListwiseRerank.from_llm(
                llm=ChatOpenAI(model=self.gpt_model), top_n=3
            )
            compression_retriever = ContextualCompressionRetriever(
                base_compressor=compressor, base_retriever=self.retriever
            )
            self.rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)
        else:
            self.rag_chain = create_retrieval_chain(self.retriever, question_answer_chain)


        
    def compile(self):
        """
        1- set_llm
        2- set_embeddings
        3- set_text_splitter
        4- fill_vectorstore
        5- compute_rag_chain
        """
        if self.text_splitter:
            print("text_splitter:", text_splitter.__class__.__name__)
            self.splits = self.text_splitter.split_documents(self.docs)
        else:
            print("text_splitter:", None)
            self.splits = self.docs
        #
        self.fill_vectorstore()
        self.compute_rag_chain()

    def run(self, question):
        results = self.rag_chain.invoke({"input": question})
        return results

        



# How to use the class

## LLM models
```
llm = ChatOpenAI(model="gpt-4o")
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)
```

for the models check:
https://www.promptfoo.dev/docs/providers/mistral/

```
open-mistral-7b, mistral-tiny, mistral-tiny-2312
open-mistral-nemo, open-mistral-nemo-2407, mistral-tiny-2407, mistral-tiny-latest
mistral-small-2402, mistral-small-latest
mistral-medium-2312, mistral-medium, mistral-medium-latest
mistral-large-2402
mistral-large-2407, mistral-large-latest
codestral-2405, codestral-latest
codestral-mamba-2407, open-codestral-mamba, codestral-mamba-latest
open-mixtral-8x7b, mistral-small, mistral-small-2312
open-mixtral-8x22b, open-mixtral-8x22b-2404
```

## embeddings
```
embeddings = OllamaEmbeddings(model="llama3")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = MistralAIEmbeddings(model="mistral-embed")
embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")
```

The full list are here:

https://python.langchain.com/docs/integrations/text_embedding/

https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/embeddings

## reranking
- None
- CohereRerank
- FlashrankRerank
- CrossEncoderReranker
- LLMListwiseRerank



In [8]:
Questions = [
"What is the core objective of investing in disruptive innovation according to ARK?",
"What are the significant risks associated with investing in innovation as highlighted by ARK?",
"Can you list the converging innovation platforms identified by ARK?",
"How does ARK describe the impact of Artificial Intelligence on technology’s integration into economic sectors?",
"What transformative potential does Multiomic Sequencing hold according to ARK?",
"What are the implications of declining battery technology costs as outlined by ARK?",
"How is the field of Robotics anticipated to evolve with the advancements in AI?",
"What does the ARK’s Convergence Scoring Framework illustrate about innovation platforms?",
"How do neural networks serve as a catalyst for other technologies?",
"What unique view does ARK have towards Autonomous Mobility and its market potential?",
"How do AI Chatbots contribute to the development of robotaxis?",
"What are breakthroughs in DNA Sequencing, particularly with neural networks?",
"How does the application of AI language models in robotics enhance general task completion rates?",
"In what ways are battery advances critical to the future of intelligent devices and augmented reality?",
"How do reusable rockets contribute to global connectivity?",
"What economic implications do disruptive innovations have according to ARK?",
"What are the top 10 holdings of ARK Innovation ETF (ARKK)?",
"What thematic strategies do ARK ETFs focus on?",
"What is ARK's strategy for capturing the benefits of disruptive innovation in its investment approach?",
"How does ARK ensure its investment strategies align with reality of disruptive innovation trends?",
]

In [9]:
# I just extracted 10 of 20, to save time
expected_outputs = [
    ["to capitalize on the potential growth", "transformative impact of innovative technologies and business models"],
    ["Market Risk", "Disruptive Innovation Risk", "Regulatory Risk", "Rapid Pace of Change", "Uncertainty and Unknowns", "Competitive Landscape", "Political or Legal Pressure"],
    ["Public Blockchains", "Multiomic Sequencing", "Artificial Intelligence", "Energy Storage", "Robotics"],
    ["Cancer Care","Precision Therapies","Programmable Biology"],
    ["Price-Parity", "Falling EV Prices", "Exponential Growth in Unit Sales", "Increased Market Penetration", "Disruption of Traditional Automotive Industry", "Environmental Impact"],
    ["AI-driven robots"],
    ["Increased Deployment of Robots","Growth in Robot Numbers","Enhanced Capabilities","Disruptive Innovation"],
    ["Public Blockchains","Multiomic Sequencing","Artificial Intelligence","Energy Storage","Robotics"],
    ["Automation and Efficiency","Data Processing and Analysis","Enhancing Capabilities of Existing Technologies","Innovation in Healthcare","Development of Intelligent Devices","Facilitating New Business Models"],
    ["costs fall by a constant percentage"],
]

# test

In [10]:
qa = QA("Investment-Case-For-Disruptive-Innovation.pdf")

number of pages: 22


In [11]:
qa.compile()

text_splitter: None
rerank_method: None


In [12]:
answer = qa.run(Questions[9])
# answer
answer["answer"]

"ARK Investment Management has a unique and optimistic view towards Autonomous Mobility, seeing it as a transformative force in the market. According to ARK, the declining costs of advanced battery technology will enable a variety of new form factors and business models. They believe Autonomous Mobility systems will significantly reduce the cost of transportation, making individual car ownership less common and increasing the velocity of e-commerce.\n\nKey points of ARK's view include:\n- The cost of getting people and goods from place to place will collapse.\n- Electric drivetrains' declining costs will unlock micro-mobility and aerial systems, like flying taxis.\n- Business models enabled by these technologies will transform urban landscapes.\n- Autonomy will reduce costs associated with taxi, delivery, and surveillance services by an order of magnitude.\n- Large-scale stationary batteries will transform the energy sector, shifting from liquid fuel to electricity and pushing generati

# Evaluation Metrics
More metrics here: https://github.com/confident-ai/deepeval

In [40]:
test_case = LLMTestCase(
    input = Questions[0],
    actual_output=answer["answer"],
    retrieval_context=expected_outputs[0]
)


In [41]:
answer["answer"]

'ARK Investment Management LLC views autonomous mobility as a highly disruptive innovation with significant market potential. They believe that the adoption of autonomous electric vehicles (EVs) could markedly disrupt the US auto loan industry. Key points of their analysis include:\n\n1. **Interest Rate Impact**: Interest rate hikes over the past three years have increased new vehicle monthly car loan payments by approximately 27%, leading to a record high in the number of subprime auto loans delinquent by 60+ days.\n\n2. **Cost Reduction**: According to Wright’s Law, the costs associated with electric vehicle production should continue to fall as production scales up. This would make EVs more affordable over time.\n\n3. **Shift to Electric Platforms**: As EV prices decrease, more miles are expected to be driven on electric platforms, which could reduce the value of gas-powered vehicles.\n\n4. **Financial Risk**: The significant amount of auto loans—approximately $1.6 trillion—currentl

## Correctness

In [42]:
correctness_metric = GEval(
    name="Correctness",
    model="gpt-4o-mini",
    evaluation_params=[
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=[
        "Determine whether the actual output is factually correct based on the expected output."
    ],
)


In [43]:
test_case_correctness = LLMTestCase(
    input=Questions[0],
    expected_output=expected_outputs[0],
    actual_output=answer["answer"],
)


In [44]:
correctness_metric.measure(test_case_correctness)
print(correctness_metric.score)


Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.23986455696488082


## Faithfulness

In [45]:
faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=False
)


In [46]:
faithfulness_metric.measure(test_case)
print(faithfulness_metric.score)
print(faithfulness_metric.reason)


Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

1.0
None


## ContextualRelevancy

In [47]:
relevance_metric = ContextualRelevancyMetric(
    threshold=1,
    model="gpt-4o-mini",
    include_reason=True
)


In [48]:
relevance_metric.measure(test_case)
print(relevance_metric.score)
print(relevance_metric.reason)


Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.0
The score is 0.00 because the context fails to provide any specific details regarding ARK's core objective of investing in disruptive innovation, only mentioning general concepts like 'potential growth' and 'transformative impact' without addressing ARK's perspective.


## AnswerRelevancy

In [51]:
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")


In [52]:
answer_relevancy_metric.measure(test_case)
print(answer_relevancy_metric.score)
print(answer_relevancy_metric.reason)


Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.6153846153846154
The score is 0.62 because several statements focused on interest rates and auto loans, which are not directly related to the core objective of investing in disruptive innovation. While the output provided some relevant information, the presence of multiple irrelevant points limited the overall effectiveness, preventing a higher score.


In [53]:
evaluate([test_case], [answer_relevancy_metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 1 test case(s) in parallel: |███████████████████████████████████████████████████████████████████████████████████████████████|100% (1/1) [Time Taken: 00:06,  6.39s/test case]



Metrics Summary

  - ✅ Answer Relevancy (score: 0.75, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.75 because while the output contains valuable insights about investing in disruptive innovation, it includes several irrelevant statements that detract from directly addressing ARK's core objective. These off-topic remarks prevent the score from being higher, but the relevant content still provides a solid foundation for understanding the primary focus., error: None)

For test case:

  - input: What is the core objective of investing in disruptive innovation according to ARK?
  - actual output: ARK Investment Management LLC views autonomous mobility as a highly disruptive innovation with significant market potential. They believe that the adoption of autonomous electric vehicles (EVs) could markedly disrupt the US auto loan industry. Key points of their analysis include:

1. **Interest Rate Impact**: Interest rate hikes over the past three years h

✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

[TestResult(success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=0.75, reason="The score is 0.75 because while the output contains valuable insights about investing in disruptive innovation, it includes several irrelevant statements that detract from directly addressing ARK's core objective. These off-topic remarks prevent the score from being higher, but the relevant content still provides a solid foundation for understanding the primary focus.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00052635, verbose_logs='Statements:\n[\n    "ARK Investment Management LLC views autonomous mobility as a highly disruptive innovation with significant market potential.",\n    "The adoption of autonomous electric vehicles (EVs) could markedly disrupt the US auto loan industry.",\n    "Interest rate hikes over the past three years have increased new vehicle monthly car loan payments by approximately 27%.",\n    "There

## more evaluations
- Contextual Recall
- Contextual Precision
- RAGAS
- Hallucination
- Toxicity
- Bias


It is also possibel to define custome functions:

https://github.com/NirDiamant/RAG_Techniques/blob/main/evaluation/evalute_rag.py

# Configurations

## Config 1 - base model

In [54]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


In [ ]:
# download from:
# https://www.morganstanley.com/content/dam/msdotcom/en/wealth-investmentsolutions/pdfs/uma/aki-1.pdf
qa = QA("Investment-Case-For-Disruptive-Innovation.pdf")


number of pages: 22


In [56]:
qa.compile()

text_splitter: None
rerank_method: None


In [57]:
answer = qa.run(Questions[0])
print(answer["answer"])


I don't have that information based on the provided context.


In [58]:
test_case_list = []
for i in range(10):
    print(i)
    answer = qa.run(Questions[i])
    test_case = LLMTestCase(
        input = Questions[i],
        actual_output=answer["answer"],
        retrieval_context=expected_outputs[i]
    )
    test_case_list.append(test_case)

#

res = evaluate(test_case_list, [faithfulness_metric, relevance_metric, answer_relevancy_metric])


0
1
2
3
4
5
6
7
8
9


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 10 test case(s) in parallel: |████████████████████████████████████████████████████████████████████████████████████████████|100% (10/10) [Time Taken: 00:10,  1.04s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: None, error: None)
  - ❌ Contextual Relevancy (score: 0.0, threshold: 1.0, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the context includes topics like 'Public Blockchains' and 'Multiomic Sequencing', which do not relate to the 'converging innovation platforms identified by ARK'., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the output directly addresses the input question about the converging innovation platforms identified by ARK with no irrelevant statements., error: None)

For test case:

  - input: Can you list the converging innovation platforms identified by ARK?
  - actual output: According to ARK Investment Management LLC, the five converging innovation platforms are:

1. Public Blockchains
2. Multiomic Sequencing
3. Artific

✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

## Config 2

In [59]:
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)



In [60]:
qa = QA("Investment-Case-For-Disruptive-Innovation.pdf")
qa.set_text_splitter(text_splitter)


number of pages: 22


In [61]:
qa.compile()

text_splitter: CharacterTextSplitter
rerank_method: None


In [62]:
answer = qa.run(Questions[0])
print(answer["answer"])


The provided context does not explicitly state the core objective of investing in disruptive innovation according to ARK Investment Management LLC. If you have more specific information or additional context, please provide it.


In [63]:
test_case_list = []
for i in range(10):
    print(i)
    answer = qa.run(Questions[i])
    test_case = LLMTestCase(
        input = Questions[i],
        actual_output=answer["answer"],
        retrieval_context=expected_outputs[i]
    )
    test_case_list.append(test_case)

#

res = evaluate(test_case_list, [faithfulness_metric, relevance_metric, answer_relevancy_metric])


0
1
2
3
4
5
6
7
8
9


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 10 test case(s) in parallel: |████████████████████████████████████████████████████████████████████████████████████████████|100% (10/10) [Time Taken: 00:13,  1.31s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: None, error: None)
  - ❌ Contextual Relevancy (score: 0.0, threshold: 1.0, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because none of the contexts listed, such as 'Public Blockchains' and 'Multiomic Sequencing', provide any information about the converging innovation platforms identified by ARK., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the output directly addresses the input without any irrelevant statements, providing focused and relevant information about the converging innovation platforms identified by ARK., error: None)

For test case:

  - input: Can you list the converging innovation platforms identified by ARK?
  - actual output: The converging innovation platforms identified by ARK are:

1. Public Blockchains
2. Multiomic Sequ

✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

## Config 3

In [64]:
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

rerank_method = "CohereRerank"


In [65]:
qa = QA("Investment-Case-For-Disruptive-Innovation.pdf")
qa.set_text_splitter(text_splitter)
qa.rerank_method = rerank_method
qa.compile()

number of pages: 22
text_splitter: CharacterTextSplitter
rerank_method: CohereRerank


/tmp/ipykernel_488/2570518993.py:52: LangChainDeprecationWarning: The class `CohereRerank` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the langchain-cohere package and should be used instead. To use it run `pip install -U langchain-cohere` and import as `from langchain_cohere import CohereRerank`.
  compressor = CohereRerank(model="rerank-english-v3.0")


In [66]:
answer = qa.run(Questions[0])
print(answer["answer"])


The core objective of investing in disruptive innovation, according to ARK Investment Management LLC, is to capitalize on the growth potential of technologies and innovations that have the potential to disrupt existing markets and create new ones. However, it is important to note that their forecasts are inherently limited and cannot be relied upon as definitive investment advice. Past performance is not indicative of future results.


In [68]:
test_case_list = []
for i in range(10):
    print(i)
    answer = qa.run(Questions[i])
    test_case = LLMTestCase(
        input = Questions[i],
        actual_output=answer["answer"],
        retrieval_context=expected_outputs[i]
    )
    test_case_list.append(test_case)

#

res = evaluate(test_case_list, [faithfulness_metric, relevance_metric, answer_relevancy_metric])


0
1
2
3
4
5
6
7
8
9


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 10 test case(s) in parallel: |████████████████████████████████████████████████████████████████████████████████████████████|100% (10/10) [Time Taken: 01:35,  9.53s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: None, error: None)
  - ❌ Contextual Relevancy (score: 0.0, threshold: 1.0, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the context 'AI-driven robots' does not address the implications of declining battery technology costs as outlined by ARK., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the output directly addresses the implications of declining battery technology costs as outlined by ARK, with no irrelevant statements present., error: None)

For test case:

  - input: What are the implications of declining battery technology costs as outlined by ARK?
  - actual output: The declining costs of battery technology have significant implications as outlined by ARK Investment Management LLC. Here are the key points:

1. **Price-Parity with

✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

## Config 4

In [69]:
# percentile
# Standard Deviation
# Interquartile
# Gradient

text_splitter = SemanticChunker(
    OpenAIEmbeddings(), breakpoint_threshold_type="percentile"
)


rerank_method = "FlashrankRerank"


In [70]:
qa = QA("Investment-Case-For-Disruptive-Innovation.pdf")
qa.set_text_splitter(text_splitter)
qa.rerank_method = rerank_method
qa.compile()

number of pages: 22
text_splitter: SemanticChunker
rerank_method: FlashrankRerank


In [71]:
test_case_list = []
for i in range(10):
    print(i)
    answer = qa.run(Questions[i])
    test_case = LLMTestCase(
        input = Questions[i],
        actual_output=answer["answer"],
        retrieval_context=expected_outputs[i]
    )
    test_case_list.append(test_case)

#

res = evaluate(test_case_list, [faithfulness_metric, relevance_metric, answer_relevancy_metric])


0
1
2
3
4
5
6
7
8
9


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 10 test case(s) in parallel: |████████████████████████████████████████████████████████████████████████████████████████████|100% (10/10) [Time Taken: 00:17,  1.78s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: None, error: None)
  - ❌ Contextual Relevancy (score: 0.0, threshold: 1.0, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the context solely discusses specific technologies like 'Public Blockchains' and 'Multiomic Sequencing', which are unrelated to the requested information about converging innovation platforms identified by ARK., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the output directly addresses the question about the converging innovation platforms identified by ARK without any irrelevant statements., error: None)

For test case:

  - input: Can you list the converging innovation platforms identified by ARK?
  - actual output: According to ARK Investment Management LLC, the five converging innovation platforms are:

1. Public

✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

In [73]:
!pip freeze > reqs.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
